# Carla Gym Environment

In [1]:
# Import libraries
import carla
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import time

In [2]:
# Define environment
class CarlaGymEnv(gym.Env):
    """
    Defines a Carla environment, supported by gym, with:
    - 
    """
    # Run when initializing
    def __init__(self, host="172.23.48.1", port=2000):
        # Connect to Carla
        self.client = carla.Client(host, port)
        self.client.set_timeout(10.0)
        self.world = self.client.get_world()

        self.blueprint_library = self.world.get_blueprint_library()

        # Action space with steering
        self.action_space = spaces.Box(
            low=np.array([-1.0]),
            high=np.array([1.0]),
            dtype=np.float32
        )

        # Frontal camera observation space
        self.image_height = 66
        self.image_width = 200

        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=(self.image_height, self.image_width, 3),
            dtype=np.uint8
        )

        # Variables
        self.vehicle = None
        self.camera = None
        self.image = None

        # Track interventions
        self.interventions = 0
        self.start_time = None

        # Activating functions
        self._enable_env_settings()

    # Set setting
    def _enable_env_settings(self):
        """
        Apply settings to Carla environment.
        """
        settings = self.world.get_settings()
        settings.synchronous_mode = True
        settings.fixed_delta_seconds = 0.05
        self.world.apply_settings(settings)

    # Set camera
    def _employ_camera(self):
        """
        Define the camera used for the vehicle
        """
        # Get and customize camera blueprint
        camera_bp = self.blueprint_library.find("sensor.camera.rgb")
        camera_bp.set_attribute("image_size_x", str(self.image_width))
        camera_bp.set_attribute("image_size_y", str(self.image_height))

        # Position camera
        transform = carla.Transform(
            carla.Location(x=1.5, z=1.4), # Slightly forward and above hood
            carla.Rotation(pitch=-5.0) # Slightly tilted downward
        )

        # Create camera
        self.camera = self.world.spawn_actor(
            camera_bp,
            transform,
            attach_to=self.vehicle
        )

    # Set vehicle
    def _employ_vehicle(self):
        """
        Define vehicle used in simulation
        """
        # Vehicle spawn point
        spawn_point = self.world.get_map().get_spawn_points()[0]

        # Vehicle blueprint
        vehicle_bp = self.blueprint_library.filter("vehicle.tesla.model3")[0]

        # Create vehicle
        self.vehicle = self.world.spawn_actor(vehicle_bp, spawn_point)

        # Inserting camera into vehicle
        self._employ_camera()

    # Reset environment
    def reset(self):
        """
        Reset Carla gym environment.
        """
        # Cleanup environment
        self.close()

        # Employ vehicle
        self._employ_vehicle()

        # Atualize environment
        self.world.tick()

        # Reset interventions
        self.interventions = 0

        # Set environment initial time
        self.start_time = time.time()

    # Clean environment
    def close(self):
        """
        Cleanup Carla gym environment.
        """
        actors = [self.camera, self.vehicle]
        for actor in actors:
            if actor:
                actor.destroy()

/home/luiscastro/dev/projects/Autonomous-Vehicle/venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/luiscastro/dev/projects/Autonomous-Vehicle/venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


# Test